# 01 — Análise Exploratória de Dados (EDA)

**Projeto:** Coffee Retail MLOps  
EDA completa para guiar as decisões de modelagem.  
**Dataset:** [Kaggle — Coffee Sales](https://www.kaggle.com/datasets/ihelon/coffee-sales)

---

## Índice

1. [Setup — Imports, Paths e Carga de Dados](#1-setup)
2. [Análise Univariada](#2-univariada)
3. [Análise Temporal](#3-temporal)
4. [Análise de Produto](#4-produto)
5. [Análise de Clientes Identificados](#5-clientes)
6. [Decomposição de Séries Temporais](#6-decomposicao)
7. [Análise de Balanceamento para Churn](#7-churn-balance)
8. [Decision Log](#8-decision-log)

---

## 1 — Setup <a id="1-setup"></a>

Carrega o dataset do Kaggle 

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import warnings
import sys
from pathlib import Path

# ── Third-party ──────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# ── Project root ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_CSV = PROJECT_ROOT / 'data' / 'raw' / 'coffee_sales.csv'
PROCESSED_PARQUET = PROJECT_ROOT / 'data' / 'processed' / 'transactions.parquet'

print(f'Project root             : {PROJECT_ROOT}')
print(f'Raw CSV exists           : {RAW_CSV.exists()}')
print(f'Processed parquet exists : {PROCESSED_PARQUET.exists()}')

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA — processed Parquet preferred, raw CSV fallback
# Raises FileNotFoundError if neither file is present.
# ─────────────────────────────────────────────────────────────────────────────

if PROCESSED_PARQUET.exists():
    df_raw = pd.read_parquet(PROCESSED_PARQUET)
    df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])
    print(f'  Loaded processed Parquet: {len(df_raw):,} rows')
elif RAW_CSV.exists():
    df_raw = pd.read_csv(RAW_CSV, parse_dates=['datetime'])
    df_raw['card'] = df_raw['card'].fillna('anonymous')
    print(f'  Loaded raw CSV: {len(df_raw):,} rows')
else:
    raise FileNotFoundError(
        'Dataset não encontrado.\n'
        f'  Esperado (processado) : {PROCESSED_PARQUET}\n'
        f'  Esperado (raw)        : {RAW_CSV}\n'
        'Baixe o dataset em https://www.kaggle.com/datasets/ihelon/coffee-sales '
        'e execute src/data/ingestion.py e src/data/preprocessing.py antes de rodar este notebook.'
    )

print(f'Date range : {df_raw["datetime"].min().date()} → {df_raw["datetime"].max().date()}')
print(f'Columns    : {list(df_raw.columns)}')
df_raw.head(10)

  Loaded processed Parquet: 3,882 rows
Date range : 2024-03-01 → 2025-03-23
Columns    : ['date', 'datetime', 'cash_type', 'card', 'money', 'coffee_name', 'hour', 'day_of_week', 'week_of_year', 'month', 'year', 'is_weekend', 'time_of_day_bucket']


,date,datetime,cash_type,card,money,coffee_name,hour,day_of_week,week_of_year,month,year,is_weekend,time_of_day_bucket
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte,10,4,9,3,2024,False,morning
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate,12,4,9,3,2024,False,afternoon
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate,12,4,9,3,2024,False,afternoon
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano,13,4,9,3,2024,False,afternoon
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte,13,4,9,3,2024,False,afternoon
5,2024-03-01,2024-03-01 15:39:47.726,card,ANON-0000-0000-0005,33.8,Americano with Milk,15,4,9,3,2024,False,afternoon
6,2024-03-01,2024-03-01 16:19:02.756,card,ANON-0000-0000-0006,38.7,Hot Chocolate,16,4,9,3,2024,False,afternoon
7,2024-03-01,2024-03-01 18:39:03.580,card,ANON-0000-0000-0007,33.8,Americano with Milk,18,4,9,3,2024,False,evening
8,2024-03-01,2024-03-01 19:22:01.762,card,ANON-0000-0000-0008,38.7,Cocoa,19,4,9,3,2024,False,evening
9,2024-03-01,2024-03-01 19:23:15.887,card,ANON-0000-0000-0008,33.8,Americano with Milk,19,4,9,3,2024,False,evening


In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# DERIVED VIEWS
# ─────────────────────────────────────────────────────────────────────────────

df = df_raw.copy()

# Ensure temporal features exist (they're added by preprocessing; add here if missing)
if 'hour' not in df.columns:
    df['hour']        = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month']       = df['datetime'].dt.month
    df['year']        = df['datetime'].dt.year
    df['is_weekend']  = df['day_of_week'] >= 5

# Identified customers only (excludes 'anonymous')
df_identified = df[df['card'] != 'anonymous'].copy()

# Reference date for recency calculations
REF_DATE = df['datetime'].max().normalize() + pd.Timedelta(days=1)   # day after last txn

print(f'Total transactions    : {len(df):,}')
print(f'Identified customers  : {df_identified["card"].nunique():,}')
print(f'Anonymous transactions : {(df["card"] == "anonymous").sum():,}')
print(f'Reference date        : {REF_DATE.date()}')

Total transactions    : 3,882
Identified customers  : 1,316
Anonymous transactions : 335
Reference date        : 2025-03-24


---

## 2 — Análise Univariada <a id="2-univariada"></a>

In [29]:
# ── 2.1  Distribuição de `money` ─────────────────────────────────────────────

print('=== money — estatísticas descritivas ===')
print(df['money'].describe().round(2).to_string())

fig_money = px.histogram(
    df,
    x='money',
    nbins=50,
    title=f'Distribuição do Valor de Transação (money) — Kaggle',
    labels={'money': 'Valor (R$)', 'count': 'Frequência'},
    color_discrete_sequence=['#3b82d4'],
    template='plotly_white',
)
fig_money.add_vline(
    x=df['money'].median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"Mediana: {df['money'].median():.2f}",
    annotation_position='top right',
)
fig_money.update_layout(height=400)
fig_money.show()

=== money — estatísticas descritivas ===
count    3882.00
mean       31.45
std         4.96
min        18.12
25%        27.92
50%        32.82
75%        35.76
max        40.00


In [30]:
# ── 2.2  Contagem por produto (`coffee_name`) ─────────────────────────────────

product_counts = (
    df.groupby('coffee_name')['money']
    .agg(volume='count', revenue='sum')
    .sort_values('volume', ascending=False)
    .reset_index()
)
product_counts['avg_ticket'] = (product_counts['revenue'] / product_counts['volume']).round(2)

print('=== Produtos — volume, receita e ticket médio ===')
print(product_counts.to_string(index=False))

fig_products = px.bar(
    product_counts,
    x='volume',
    y='coffee_name',
    orientation='h',
    title=f'Volume de Vendas por Produto — Kaggle',
    labels={'volume': 'Nº de Transações', 'coffee_name': 'Produto'},
    color='volume',
    color_continuous_scale='Blues',
    template='plotly_white',
)
fig_products.update_layout(yaxis={'categoryorder': 'total ascending'}, height=420, showlegend=False)
fig_products.show()

=== Produtos — volume, receita e ticket médio ===
               coffee_name  volume  revenue  avg_ticket
       Americano with Milk     824 25269.12       30.67
                     Latte     806 28658.30       35.56
                 Americano     593 15437.26       26.03
                Cappuccino     517 18514.14       35.81
                   Cortado     292  7534.86       25.80
             Hot Chocolate     282 10172.46       36.07
                     Cocoa     243  8678.16       35.71
                  Espresso     152  3187.28       20.97
       Americano with milk      44  1100.00       25.00
             Irish whiskey      21   525.00       25.00
 Coffee with Irish whiskey      16   464.00       29.00
   Irish whiskey with milk      15   405.00       27.00
       Chocolate with milk      12   324.00       27.00
      Double Irish whiskey      12   336.00       28.00
Caramel with Irish whiskey       8   224.00       28.00
           Super chocolate       6   168.00       28.0

In [31]:
# ── 2.3  Distribuição de `cash_type` ─────────────────────────────────────────

cash_dist = df['cash_type'].value_counts().reset_index()
cash_dist.columns = ['cash_type', 'count']
cash_dist['pct'] = (cash_dist['count'] / cash_dist['count'].sum() * 100).round(1)

print('=== Distribuição de método de pagamento ===')
print(cash_dist.to_string(index=False))

fig_cash = px.pie(
    cash_dist,
    names='cash_type',
    values='count',
    title=f'Método de Pagamento — Kaggle',
    color_discrete_sequence=['#3b82d4', '#7c5cd8'],
    template='plotly_white',
    hole=0.35,
)
fig_cash.update_traces(textinfo='percent+label')
fig_cash.update_layout(height=350)
fig_cash.show()

=== Distribuição de método de pagamento ===
cash_type  count  pct
     card   3719 95.8
     cash    163  4.2


In [32]:
# ── 2.4  Proporção de clientes identificados vs. anônimos ────────────────────

anon_count = (df['card'] == 'anonymous').sum()
ident_txn  = len(df) - anon_count
print(f'Transações identificadas : {ident_txn:,} ({ident_txn/len(df)*100:.1f}%)')
print(f'Transações anônimas      : {anon_count:,} ({anon_count/len(df)*100:.1f}%)')
print(f'Clientes únicos          : {df_identified["card"].nunique():,}')

Transações identificadas : 3,547 (91.4%)
Transações anônimas      : 335 (8.6%)
Clientes únicos          : 1,316


---

## 3 — Análise Temporal <a id="3-temporal"></a>

In [33]:
# ── 3.1  Vendas por hora do dia ───────────────────────────────────────────────

hourly = (
    df.groupby('hour')['money']
    .agg(transactions='count', revenue='sum')
    .reset_index()
)

fig_hour = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Volume de Transações por Hora', 'Receita por Hora'),
)
fig_hour.add_trace(
    go.Bar(x=hourly['hour'], y=hourly['transactions'], marker_color='#3b82d4', name='Transações'),
    row=1, col=1,
)
fig_hour.add_trace(
    go.Bar(x=hourly['hour'], y=hourly['revenue'].round(0), marker_color='#7c5cd8', name='Receita'),
    row=1, col=2,
)
fig_hour.update_layout(
    title_text=f'Padrão Horário — Kaggle',
    template='plotly_white',
    height=380,
    showlegend=False,
)
fig_hour.update_xaxes(title_text='Hora do Dia', dtick=2)
fig_hour.show()

peak_hour = int(hourly.loc[hourly['transactions'].idxmax(), 'hour'])
print(f'Hora de pico : {peak_hour}h ({hourly["transactions"].max():,} transações)')

Hora de pico : 10h (363 transações)


In [34]:
# ── 3.2  Vendas por dia da semana ─────────────────────────────────────────────

DOW_NAMES = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']

dow = (
    df.groupby('day_of_week')['money']
    .agg(transactions='count', revenue='sum')
    .reset_index()
)
dow['day_name'] = dow['day_of_week'].map(lambda x: DOW_NAMES[x])

fig_dow = px.bar(
    dow,
    x='day_name',
    y='transactions',
    title=f'Volume por Dia da Semana — Kaggle',
    labels={'transactions': 'Nº de Transações', 'day_name': 'Dia da Semana'},
    color='transactions',
    color_continuous_scale='Blues',
    template='plotly_white',
    category_orders={'day_name': DOW_NAMES},
)
fig_dow.update_layout(height=380, showlegend=False)
fig_dow.show()

peak_dow = DOW_NAMES[int(dow.loc[dow['transactions'].idxmax(), 'day_of_week'])]
print(f'Dia de maior volume : {peak_dow}')

Dia de maior volume : Ter


In [35]:
# ── 3.3  Vendas por mês ───────────────────────────────────────────────────────

monthly = (
    df.groupby(['year', 'month'])['money']
    .agg(transactions='count', revenue='sum')
    .reset_index()
)
monthly['period'] = monthly.apply(
    lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1
)

fig_monthly = make_subplots(rows=2, cols=1,
    subplot_titles=('Transações Mensais', 'Receita Mensal (R$)'))

fig_monthly.add_trace(
    go.Bar(x=monthly['period'], y=monthly['transactions'], marker_color='#3b82d4', name='Transações'),
    row=1, col=1,
)
fig_monthly.add_trace(
    go.Bar(x=monthly['period'], y=monthly['revenue'].round(0), marker_color='#7c5cd8', name='Receita'),
    row=2, col=1,
)
fig_monthly.update_layout(
    title_text=f'Evolução Mensal — Kaggle',
    template='plotly_white',
    height=550,
    showlegend=False,
)
fig_monthly.show()

---

## 4 — Análise de Produto <a id="4-produto"></a>

In [36]:
# ── 4.1  Top e Bottom produtos por volume ─────────────────────────────────────

top_n = min(5, len(product_counts))
top_products   = product_counts.head(top_n)
bottom_products = product_counts.tail(top_n)

fig_top_bottom = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Top {top_n} por Volume',
        f'Bottom {top_n} por Volume',
    ),
)
fig_top_bottom.add_trace(
    go.Bar(
        x=top_products['volume'],
        y=top_products['coffee_name'],
        orientation='h',
        marker_color='#3b82d4',
        name='Top',
    ),
    row=1, col=1,
)
fig_top_bottom.add_trace(
    go.Bar(
        x=bottom_products['volume'],
        y=bottom_products['coffee_name'],
        orientation='h',
        marker_color='#e05252',
        name='Bottom',
    ),
    row=1, col=2,
)
fig_top_bottom.update_layout(
    title_text=f'Top vs. Bottom Produtos por Volume — Kaggle',
    template='plotly_white',
    height=380,
    showlegend=False,
)
fig_top_bottom.show()

In [37]:
# ── 4.2  Top e Bottom produtos por receita total ─────────────────────────────

by_revenue = product_counts.sort_values('revenue', ascending=False).reset_index(drop=True)
top_rev    = by_revenue.head(top_n)
bottom_rev = by_revenue.tail(top_n)

fig_revenue = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Top {top_n} por Receita',
        f'Bottom {top_n} por Receita',
    ),
)
fig_revenue.add_trace(
    go.Bar(
        x=top_rev['revenue'].round(0),
        y=top_rev['coffee_name'],
        orientation='h',
        marker_color='#3b82d4',
        name='Top Receita',
    ),
    row=1, col=1,
)
fig_revenue.add_trace(
    go.Bar(
        x=bottom_rev['revenue'].round(0),
        y=bottom_rev['coffee_name'],
        orientation='h',
        marker_color='#e05252',
        name='Bottom Receita',
    ),
    row=1, col=2,
)
fig_revenue.update_layout(
    title_text=f'Top vs. Bottom Produtos por Receita — Kaggle',
    template='plotly_white',
    height=380,
    showlegend=False,
)
fig_revenue.show()

print('\n=== Receita total por produto ===')
for _, row in by_revenue.iterrows():
    print(f"  {row['coffee_name']:<18} R$ {row['revenue']:>8.2f}  ({row['volume']:>4} txns)")


=== Receita total por produto ===
  Latte              R$ 28658.30  ( 806 txns)
  Americano with Milk R$ 25269.12  ( 824 txns)
  Cappuccino         R$ 18514.14  ( 517 txns)
  Americano          R$ 15437.26  ( 593 txns)
  Hot Chocolate      R$ 10172.46  ( 282 txns)
  Cocoa              R$  8678.16  ( 243 txns)
  Cortado            R$  7534.86  ( 292 txns)
  Espresso           R$  3187.28  ( 152 txns)
  Americano with milk R$  1100.00  (  44 txns)
  Irish whiskey      R$   525.00  (  21 txns)
  Coffee with Irish whiskey R$   464.00  (  16 txns)
  Irish whiskey with milk R$   405.00  (  15 txns)
  Double Irish whiskey R$   336.00  (  12 txns)
  Chocolate with milk R$   324.00  (  12 txns)
  Caramel with Irish whiskey R$   224.00  (   8 txns)
  Super chocolate    R$   168.00  (   6 txns)
  Chocolate with coffee R$   150.00  (   5 txns)
  Mochaccino         R$   150.00  (   5 txns)
  Irish with chocolate R$   135.00  (   5 txns)
  Caramel with chocolate R$   112.00  (   4 txns)
  Vanilla w

---

## 5 — Análise de Clientes Identificados <a id="5-clientes"></a>

> Apenas registros com `card != anonymous` são considerados nesta seção.

In [38]:
# ── 5.1  Métricas por cliente ─────────────────────────────────────────────────

n_identified = df_identified['card'].nunique()

customer_stats = (
    df_identified
    .groupby('card')
    .agg(
        frequency=('datetime', 'count'),
        total_spend=('money', 'sum'),
        avg_ticket=('money', 'mean'),
        first_purchase=('datetime', 'min'),
        last_purchase=('datetime', 'max'),
    )
    .reset_index()
)
customer_stats['recency_days'] = (
    (REF_DATE - customer_stats['last_purchase']).dt.days
)
customer_stats['tenure_days'] = (
    (customer_stats['last_purchase'] - customer_stats['first_purchase']).dt.days
)

print(f'Clientes identificados   : {n_identified:,}')
print(f"Frequência média          : {customer_stats['frequency'].mean():.1f} compras")
print(f"Ticket médio médio        : R$ {customer_stats['avg_ticket'].mean():.2f}")
print(f"Recência mediana          : {int(customer_stats['recency_days'].median())} dias")
customer_stats.describe().round(2)

Clientes identificados   : 1,316
Frequência média          : 2.7 compras
Ticket médio médio        : R$ 31.91
Recência mediana          : 162 dias


,frequency,total_spend,avg_ticket,first_purchase,last_purchase,recency_days,tenure_days
count,1316.00,1316.00,1316.00,1316,1316,1316.00,1316.00
mean,2.70,85.29,31.91,2024-09-18 02:54:19.720821248,2024-10-03 16:28:37.847020544,170.94,15.43
min,1.00,18.12,18.12,2024-03-01 10:15:50.520000,2024-03-01 12:20:18.089000,0.00,0.00
25%,1.00,32.82,27.92,2024-06-28 15:50:49.435500032,2024-07-23 09:45:03.645750016,90.00,0.00
50%,1.00,35.76,32.82,2024-09-23 18:08:09.013499904,2024-10-12 05:15:33.896999936,162.50,0.00
75%,2.00,71.52,35.76,2024-12-07 16:12:34.668249856,2024-12-23 14:19:14.720999936,243.00,0.00
max,129.00,3785.92,38.70,2025-03-23 15:47:28.723000,2025-03-23 18:11:38.635000,387.00,382.00
std,6.81,204.99,4.64,NaN,NaN,103.18,45.32


In [39]:
# ── 5.2  Distribuição de frequência de compra ─────────────────────────────────

fig_freq = px.histogram(
    customer_stats,
    x='frequency',
    nbins=40,
    title=f'Distribuição de Frequência de Compra — Kaggle',
    labels={'frequency': 'Nº de Compras', 'count': 'Nº de Clientes'},
    color_discrete_sequence=['#3b82d4'],
    template='plotly_white',
)
fig_freq.update_layout(height=380)
fig_freq.show()

print(f"Clientes com 1 única compra : {(customer_stats['frequency'] == 1).sum():,}")
print(f"Clientes com >= 5 compras   : {(customer_stats['frequency'] >= 5).sum():,}")

Clientes com 1 única compra : 771
Clientes com >= 5 compras   : 121


In [40]:
# ── 5.3  Distribuição de ticket médio ─────────────────────────────────────────

fig_ticket = px.histogram(
    customer_stats,
    x='avg_ticket',
    nbins=40,
    title=f'Distribuição de Ticket Médio por Cliente — Kaggle',
    labels={'avg_ticket': 'Ticket Médio (R$)', 'count': 'Nº de Clientes'},
    color_discrete_sequence=['#7c5cd8'],
    template='plotly_white',
)
fig_ticket.update_layout(height=380)
fig_ticket.show()

In [41]:
# ── 5.4  Distribuição de dias entre compras (inter-purchase interval) ─────────

ipi_records = []
for card, grp in df_identified.groupby('card'):
    dates = grp['datetime'].sort_values().dt.normalize().drop_duplicates()
    if len(dates) >= 2:
        diffs = dates.diff().dropna().dt.days.tolist()
        ipi_records.extend(diffs)

ipi_series = pd.Series(ipi_records, name='ipi_days')

print(f'Intervalos calculados : {len(ipi_series):,}')
print(ipi_series.describe().round(1).to_string())

fig_ipi = px.histogram(
    ipi_series,
    x='ipi_days',
    nbins=60,
    range_x=[0, 120],
    title=f'Intervalo Entre Compras (inter-purchase interval) — Kaggle',
    labels={'ipi_days': 'Dias entre Compras', 'count': 'Frequência'},
    color_discrete_sequence=['#3b82d4'],
    template='plotly_white',
)
fig_ipi.add_vline(
    x=ipi_series.median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"Mediana: {ipi_series.median():.0f}d",
)
fig_ipi.update_layout(height=380)
fig_ipi.show()

Intervalos calculados : 1,440
count    1440.0
mean       14.2
std        30.1
min         1.0
25%         2.0
50%         4.0
75%        13.0
max       340.0


In [42]:
# ── 5.5  Distribuição de recência (dias desde última compra) ──────────────────

fig_recency = px.histogram(
    customer_stats,
    x='recency_days',
    nbins=60,
    title=f'Distribuição de Recência (dias desde última compra) — Kaggle',
    labels={'recency_days': 'Dias desde última compra', 'count': 'Nº de Clientes'},
    color_discrete_sequence=['#7c5cd8'],
    template='plotly_white',
)
fig_recency.add_vline(
    x=customer_stats['recency_days'].median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"Mediana: {int(customer_stats['recency_days'].median())}d",
)
for threshold in [30, 60, 90]:
    fig_recency.add_vline(
        x=threshold,
        line_dash='dot',
        line_color='orange',
        annotation_text=f'{threshold}d',
        annotation_position='top right',
    )
fig_recency.update_layout(height=400)
fig_recency.show()

---

## 6 — Decomposição de Séries Temporais <a id="6-decomposicao"></a>

In [43]:
# ── 6.1  Série temporal diária — top 3 produtos ───────────────────────────────

TOP3_PRODUCTS = product_counts.head(5)['coffee_name'].tolist()
print(f'Top 5 produtos por volume: {TOP3_PRODUCTS}')

# Série diária por produto
daily_product = (
    df.groupby(['coffee_name', df['datetime'].dt.normalize()])
    ['money']
    .count()
    .reset_index()
    .rename(columns={'datetime': 'date', 'money': 'daily_sales'})
)

fig_ts = make_subplots(
    rows=len(TOP3_PRODUCTS),
    cols=1,
    subplot_titles=[f'{p} — Série Diária com MM7' for p in TOP3_PRODUCTS],
    shared_xaxes=True,
)

series_lengths = {}
for i, prod in enumerate(TOP3_PRODUCTS, start=1):
    s = daily_product[daily_product['coffee_name'] == prod].set_index('date').sort_index()

    # full daily index to expose gaps
    full_idx = pd.date_range(s.index.min(), s.index.max(), freq='D')
    s = s['daily_sales'].reindex(full_idx, fill_value=0).to_frame(name='daily_sales')
    s.index.name = 'date'

    n_days = (s.index.max() - s.index.min()).days + 1
    n_active = (s['daily_sales'] > 0).sum()
    series_lengths[prod] = {'n_days': n_days, 'n_active_days': n_active}

    mm7 = s['daily_sales'].rolling(7, min_periods=1).mean()

    fig_ts.add_trace(
        go.Scatter(
            x=s.index, y=s['daily_sales'],
            mode='lines', name=prod,
            line=dict(color='#3b82d4', width=1),
            opacity=0.5,
        ),
        row=i, col=1,
    )
    fig_ts.add_trace(
        go.Scatter(
            x=s.index, y=mm7,
            mode='lines', name=f'{prod} MM7',
            line=dict(color='#e05252', width=2),
        ),
        row=i, col=1,
    )

fig_ts.update_layout(
    title_text=f'Séries Temporais Diárias (Top 5 Produtos) — Kaggle',
    template='plotly_white',
    height=600,
    showlegend=False,
)
fig_ts.show()

print('\n=== Extensão das séries ===')
for prod, info in series_lengths.items():
    print(f"  {prod:<18}: {info['n_days']} dias calendário | {info['n_active_days']} dias com vendas")

Top 5 produtos por volume: ['Americano with Milk', 'Latte', 'Americano', 'Cappuccino', 'Cortado']



=== Extensão das séries ===
  Americano with Milk: 387 dias calendário | 316 dias com vendas
  Latte             : 388 dias calendário | 310 dias com vendas
  Americano         : 388 dias calendário | 249 dias com vendas
  Cappuccino        : 383 dias calendário | 265 dias com vendas
  Cortado           : 383 dias calendário | 173 dias com vendas


In [44]:
# ── 6.2  Decomposição STL ─────────────────────────────────────────────────────

MIN_DAYS_FOR_STL = 60   # mínimo razoável para STL com sazonalidade semanal (period=7)
stl_results = {}

try:
    from statsmodels.tsa.seasonal import STL
    stl_available = True
except ImportError:
    stl_available = False
    print('⚠️  statsmodels não instalado — pulando decomposição STL')

if stl_available:
    for prod in TOP3_PRODUCTS:
        s = daily_product[daily_product['coffee_name'] == prod].set_index('date').sort_index()
        full_idx = pd.date_range(s.index.min(), s.index.max(), freq='D')
        s = (
            daily_product[daily_product['coffee_name'] == prod]
            .set_index('date')
            .sort_index()['daily_sales']
            .reindex(full_idx, fill_value=0)
            .astype(float)
        )

        n = len(s)
        active_days = (s > 0).sum()

        if n < MIN_DAYS_FOR_STL:
            reason = f'série muito curta ({n} dias < {MIN_DAYS_FOR_STL} mínimo)'
            stl_results[prod] = {'status': 'skipped', 'reason': reason}
            print(f'  {prod:<18}: PULADO — {reason}')
            continue

        if active_days < n * 0.30:
            reason = f'muitos zeros ({active_days}/{n} dias com vendas — {active_days/n*100:.0f}%)'
            stl_results[prod] = {'status': 'skipped', 'reason': reason}
            print(f'  {prod:<18}: PULADO — {reason}')
            continue

        try:
            stl = STL(s, period=7, robust=True)
            result = stl.fit()
            stl_results[prod] = {'status': 'ok', 'result': result}

            # Plot decomposition
            fig_stl = make_subplots(
                rows=4, cols=1,
                subplot_titles=['Série Original', 'Tendência', 'Sazonalidade', 'Resíduo'],
                shared_xaxes=True,
            )
            components = [
                (result.observed,  '#3b82d4'),
                (result.trend,     '#f59e0b'),
                (result.seasonal,  '#10b981'),
                (result.resid,     '#6b7280'),
            ]
            for row_idx, (component, color) in enumerate(components, start=1):
                fig_stl.add_trace(
                    go.Scatter(x=s.index, y=component, mode='lines',
                               line=dict(color=color, width=1.5)),
                    row=row_idx, col=1,
                )
            fig_stl.update_layout(
                title_text=f'Decomposição STL — {prod} (Kaggle)',
                template='plotly_white',
                height=700,
                showlegend=False,
            )
            fig_stl.show()
            print(f'  {prod:<18}: STL OK ({n} dias)')

        except Exception as exc:
            stl_results[prod] = {'status': 'error', 'reason': str(exc)}
            print(f'  {prod:<18}: ERRO — {exc}')

print('\n=== Resumo STL ===')
for prod, info in stl_results.items():
    print(f"  {prod:<18}: {info['status']}" + (f" — {info.get('reason','')}" if info['status'] != 'ok' else ''))

  Americano with Milk: STL OK (387 dias)


  Latte             : STL OK (388 dias)


  Americano         : STL OK (388 dias)


  Cappuccino        : STL OK (383 dias)


  Cortado           : STL OK (383 dias)

=== Resumo STL ===
  Americano with Milk: ok
  Latte             : ok
  Americano         : ok
  Cappuccino        : ok
  Cortado           : ok


In [45]:
# ── 6.2.1  Decomposição STL (Estações do Ano / 3 meses) ──────────────────────────

# Mínimo razoável para STL com sazonalidade trimestral (period=91)
# Ideal ter pelo menos 2 ciclos completos (2 * 91 = 182 dias)
MIN_DAYS_FOR_STL = 182   
stl_results = {}

try:
    from statsmodels.tsa.seasonal import STL
    stl_available = True
except ImportError:
    stl_available = False
    print('⚠️  statsmodels não instalado — pulando decomposição STL')

if stl_available:
    for prod in TOP3_PRODUCTS:
        # Extração e formatação correta da série diária
        s_raw = daily_product[daily_product['coffee_name'] == prod].set_index('date').sort_index()
        full_idx = pd.date_range(s_raw.index.min(), s_raw.index.max(), freq='D')
        
        s = (
            s_raw['daily_sales']
            .reindex(full_idx, fill_value=0)
            .astype(float)
        )

        n = len(s)
        active_days = (s > 0).sum()

        if n < MIN_DAYS_FOR_STL:
            reason = f'série muito curta ({n} dias < {MIN_DAYS_FOR_STL} mínimo)'
            stl_results[prod] = {'status': 'skipped', 'reason': reason}
            print(f'  {prod:<18}: PULADO — {reason}')
            continue

        if active_days < n * 0.30:
            reason = f'muitos zeros ({active_days}/{n} dias com vendas — {active_days/n*100:.0f}%)'
            stl_results[prod] = {'status': 'skipped', 'reason': reason}
            print(f'  {prod:<18}: PULADO — {reason}')
            continue

        try:
            # period=91 captura o ciclo de 3 meses (~1 estação do ano)
            # seasonal=13 (deve ser ímpar) define a suavização da janela sazonal
            stl = STL(s, period=91, seasonal=13, robust=True)
            result = stl.fit()
            stl_results[prod] = {'status': 'ok', 'result': result}

            # Plot da decomposição
            fig_stl = make_subplots(
                rows=4, cols=1,
                subplot_titles=[
                    'Série Original', 
                    'Tendência', 
                    'Sazonalidade (Ciclo Trimestral / Estações)', 
                    'Resíduo'
                ],
                shared_xaxes=True,
            )
            components = [
                (result.observed,  '#3b82d4'),
                (result.trend,     '#f59e0b'),
                (result.seasonal,  '#10b981'),
                (result.resid,     '#6b7280'),
            ]
            
            for row_idx, (component, color) in enumerate(components, start=1):
                fig_stl.add_trace(
                    go.Scatter(x=s.index, y=component, mode='lines',
                               line=dict(color=color, width=1.5)),
                    row=row_idx, col=1,
                )
                
            fig_stl.update_layout(
                title_text=f'Decomposição STL Trimestral — {prod} (Kaggle)',
                template='plotly_white',
                height=700,
                showlegend=False,
            )
            fig_stl.show()
            print(f'  {prod:<18}: STL OK ({n} dias)')

        except Exception as exc:
            stl_results[prod] = {'status': 'error', 'reason': str(exc)}
            print(f'  {prod:<18}: ERRO — {exc}')

print('\n=== Resumo STL ===')
for prod, info in stl_results.items():
    print(f"  {prod:<18}: {info['status']}" + (f" — {info.get('reason','')}" if info['status'] != 'ok' else ''))

  Americano with Milk: STL OK (387 dias)


  Latte             : STL OK (388 dias)


  Americano         : STL OK (388 dias)


  Cappuccino        : STL OK (383 dias)


  Cortado           : STL OK (383 dias)

=== Resumo STL ===
  Americano with Milk: ok
  Latte             : ok
  Americano         : ok
  Cappuccino        : ok
  Cortado           : ok


In [46]:
# ── 6.3  Produtos elegíveis para forecasting individual ───────────────────────

MIN_ACTIVE_DAYS_FOR_FORECAST = 60  # critério: >= 60 dias com pelo menos 1 venda

all_product_series = (
    df.groupby(['coffee_name', df['datetime'].dt.normalize()])
    ['money'].count()
    .reset_index()
    .rename(columns={'datetime': 'date', 'money': 'daily_sales'})
)

forecast_eligible = []
for prod, grp in all_product_series.groupby('coffee_name'):
    active_days = (grp['daily_sales'] > 0).sum()
    if active_days >= MIN_ACTIVE_DAYS_FOR_FORECAST:
        forecast_eligible.append(prod)

print(f'Produtos elegíveis para forecasting individual (>= {MIN_ACTIVE_DAYS_FOR_FORECAST} dias ativos):')
for p in forecast_eligible:
    grp = all_product_series[all_product_series['coffee_name'] == p]
    active = int((grp['daily_sales'] > 0).sum())
    print(f'    {p:<18} — {active} dias ativos')

not_eligible = [
    p for p in product_counts['coffee_name'].tolist()
    if p not in forecast_eligible
]
if not_eligible:
    print('\nProdutos excluídos do forecasting individual:')
    for p in not_eligible:
        grp = all_product_series[all_product_series['coffee_name'] == p]
        active = int((grp['daily_sales'] > 0).sum())
        print(f'    {p:<18} — {active} dias ativos')

Produtos elegíveis para forecasting individual (>= 60 dias ativos):
    Americano          — 249 dias ativos
    Americano with Milk — 316 dias ativos
    Cappuccino         — 265 dias ativos
    Cocoa              — 141 dias ativos
    Cortado            — 173 dias ativos
    Espresso           — 110 dias ativos
    Hot Chocolate      — 160 dias ativos
    Latte              — 310 dias ativos

Produtos excluídos do forecasting individual:
    Americano with milk — 26 dias ativos
    Irish whiskey      — 15 dias ativos
    Coffee with Irish whiskey — 14 dias ativos
    Irish whiskey with milk — 12 dias ativos
    Chocolate with milk — 12 dias ativos
    Double Irish whiskey — 9 dias ativos
    Caramel with Irish whiskey — 6 dias ativos
    Super chocolate    — 6 dias ativos
    Chocolate with coffee — 4 dias ativos
    Mochaccino         — 5 dias ativos
    Irish with chocolate — 4 dias ativos
    Caramel with chocolate — 4 dias ativos
    Vanilla with Irish whiskey — 3 dias ativos
   

---

## 7 — Análise de Balanceamento para Churn <a id="7-churn-balance"></a>

In [47]:
# ── 7.1  Proporção de clientes inativos por janela ────────────────────────────

churn_windows = [30, 60, 90]
balance_report = []

for w in churn_windows:
    churned = (customer_stats['recency_days'] > w).sum()
    active  = len(customer_stats) - churned
    ratio   = churned / len(customer_stats)
    minority_ratio = min(churned, active) / len(customer_stats)
    balance_report.append({
        'window_days':     w,
        'churned':         churned,
        'active':          active,
        'churn_rate_pct':  round(ratio * 100, 1),
        'minority_pct':    round(minority_ratio * 100, 1),
    })

df_balance = pd.DataFrame(balance_report)

print('=== Balanceamento de classes por janela de churn ===')
print(df_balance.to_string(index=False))

fig_balance = px.bar(
    df_balance,
    x='window_days',
    y=['churned', 'active'],
    barmode='stack',
    title=f'Balanceamento de Classes — Kaggle',
    labels={'window_days': 'Janela de Inatividade (dias)', 'value': 'Nº de Clientes', 'variable': 'Status'},
    color_discrete_map={'churned': '#e05252', 'active': '#3b82d4'},
    template='plotly_white',
)
fig_balance.update_layout(height=380)
fig_balance.show()

=== Balanceamento de classes por janela de churn ===
 window_days  churned  active  churn_rate_pct  minority_pct
          30     1179     137            89.6          10.4
          60     1095     221            83.2          16.8
          90      985     331            74.8          25.2


In [48]:
# ── 7.2  Seleção da janela ótima ──────────────────────────────────────────────

# Heurística: janela que mantém minority class ratio entre 20-50% (class balance razoável)
# Fallback: janela com minority ratio mais próximo de 30%

IDEAL_MINORITY_MIN = 20.0
IDEAL_MINORITY_MAX = 50.0

balanced_candidates = df_balance[
    (df_balance['minority_pct'] >= IDEAL_MINORITY_MIN) &
    (df_balance['minority_pct'] <= IDEAL_MINORITY_MAX)
]

if len(balanced_candidates) > 0:
    # prefer shortest window that achieves balance (less aggressive label)
    best_row = balanced_candidates.sort_values('window_days').iloc[0]
    CHURN_WINDOW_DAYS = int(best_row['window_days'])
    churn_balance_note = (
        f"Janela de {CHURN_WINDOW_DAYS} dias → "
        f"{best_row['churn_rate_pct']}% churn "
        f"(minority {best_row['minority_pct']}% — balanceamento adequado para classificação supervisionada)"
    )
else:
    # all windows are imbalanced — pick closest to 30% minority
    df_balance['dist_30'] = (df_balance['minority_pct'] - 30.0).abs()
    best_row = df_balance.sort_values('dist_30').iloc[0]
    CHURN_WINDOW_DAYS = int(best_row['window_days'])
    churn_balance_note = (
        f"Nenhuma janela produz classe minoritária >= {IDEAL_MINORITY_MIN}%. "
        f"Janela de {CHURN_WINDOW_DAYS} dias selecionada como menos desequilibrada "
        f"(minority {best_row['minority_pct']}%). Recomendar anomaly_detection ou SMOTE."
    )

print(f'\n  CHURN_WINDOW_DAYS = {CHURN_WINDOW_DAYS}')
print(f'   {churn_balance_note}')

# Abordagem recomendada
minority_pct = float(best_row['minority_pct'])
if minority_pct >= 20.0:
    CHURN_APPROACH = 'classification'
    churn_approach_note = (
        f'Classe minoritária = {minority_pct:.1f}% — balanceamento aceitável para '
        'classificação binária supervisionada com scale_pos_weight ou SMOTE'
    )
else:
    CHURN_APPROACH = 'anomaly_detection'
    churn_approach_note = (
        f'Classe minoritária = {minority_pct:.1f}% — desequilíbrio severo, '
        'recomendado abordagem de detecção de anomalia (Isolation Forest / One-Class SVM)'
    )

print(f'\n  CHURN_APPROACH = {CHURN_APPROACH}')
print(f'   {churn_approach_note}')


  CHURN_WINDOW_DAYS = 90
   Janela de 90 dias → 74.8% churn (minority 25.2% — balanceamento adequado para classificação supervisionada)

  CHURN_APPROACH = classification
   Classe minoritária = 25.2% — balanceamento aceitável para classificação binária supervisionada com scale_pos_weight ou SMOTE


In [49]:
# ── 7.3  Viabilidade de clustering RFM ───────────────────────────────────────

MIN_CUSTOMERS_FOR_CLUSTERING = 50

CLUSTERING_VIABLE = n_identified >= MIN_CUSTOMERS_FOR_CLUSTERING
clustering_note = (
    f'{n_identified} clientes identificados — '
    + ('suficiente' if CLUSTERING_VIABLE else 'insuficiente')
    + f' para clustering RFM (threshold: {MIN_CUSTOMERS_FOR_CLUSTERING})'
)

print(f'  CLUSTERING_VIABLE = {CLUSTERING_VIABLE}')
print(f'   {clustering_note}')

# Model candidates for forecasting — favor ML with lags for short/irregular series
all_series_ok = all(info.get('status') == 'ok' for info in stl_results.values())

if len(forecast_eligible) == 0:
    FORECAST_MODEL_CANDIDATES = ['XGBoost com features de lag (lag-1, lag-7, lag-14)']
    forecast_model_note = (
        'Séries muito curtas ou irregulares — Prophet descartado. '
        'Priorizar XGBoost com features de lag + rolling mean'
    )
elif all_series_ok:
    FORECAST_MODEL_CANDIDATES = [
        'Prophet (sazonalidade semanal + anual)',
        'XGBoost com features de lag',
        'Seasonal Naive (baseline)',
    ]
    forecast_model_note = (
        'Séries longas e regulares — Prophet viável. '
        'Comparar com XGBoost com features de lag'
    )
else:
    FORECAST_MODEL_CANDIDATES = [
        'XGBoost com features de lag (lag-1, lag-7, lag-14)',
        'Seasonal Naive (baseline)',
        'Prophet (apenas produtos com série >= 90 dias)',
    ]
    forecast_model_note = (
        'Séries mistas — Prophet condicional a comprimento. '
        'XGBoost como candidato principal; Seasonal Naive como baseline'
    )

print(f'\n  FORECAST_MODEL_CANDIDATES = {FORECAST_MODEL_CANDIDATES}')
print(f'   {forecast_model_note}')

  CLUSTERING_VIABLE = True
   1316 clientes identificados — suficiente para clustering RFM (threshold: 50)

  FORECAST_MODEL_CANDIDATES = ['Prophet (sazonalidade semanal + anual)', 'XGBoost com features de lag', 'Seasonal Naive (baseline)']
   Séries longas e regulares — Prophet viável. Comparar com XGBoost com features de lag


---

## 8 — Decision Log <a id="8-decision-log"></a>

> Todas as decisões abaixo são derivadas dos dados analisados nas seções anteriores.

In [50]:
# ── Consolida todos os valores do Decision Log ────────────────────────────────

# Garante que variáveis existem mesmo se células anteriores foram puladas
try:
    _ = CHURN_WINDOW_DAYS
except NameError:
    CHURN_WINDOW_DAYS = 60
    churn_balance_note = 'Valor padrão — reanalisar com dados reais'

try:
    _ = CHURN_APPROACH
except NameError:
    CHURN_APPROACH = 'classification'
    churn_approach_note = 'Valor padrão — reanalisar com dados reais'

try:
    _ = forecast_eligible
except NameError:
    forecast_eligible = []

try:
    _ = FORECAST_MODEL_CANDIDATES
except NameError:
    FORECAST_MODEL_CANDIDATES = ['XGBoost com features de lag']
    forecast_model_note = 'Valor padrão — reanalisar com dados reais'

try:
    _ = CLUSTERING_VIABLE
except NameError:
    CLUSTERING_VIABLE = n_identified >= 50
    clustering_note = f'{n_identified} clientes identificados'

print('Decision Log — valores consolidados')
print(f'  CHURN_WINDOW_DAYS            = {CHURN_WINDOW_DAYS}')
print(f'  FORECAST_PRODUCTS            = {forecast_eligible}')
print(f'  FORECAST_MODEL_CANDIDATES    = {FORECAST_MODEL_CANDIDATES}')
print(f'  CHURN_APPROACH               = {CHURN_APPROACH}')
print(f'  MIN_IDENTIFIED_CUSTOMERS     = {n_identified}')
print(f'  CLUSTERING_VIABLE            = {CLUSTERING_VIABLE}')

Decision Log — valores consolidados
  CHURN_WINDOW_DAYS            = 90
  FORECAST_PRODUCTS            = ['Americano', 'Americano with Milk', 'Cappuccino', 'Cocoa', 'Cortado', 'Espresso', 'Hot Chocolate', 'Latte']
  FORECAST_MODEL_CANDIDATES    = ['Prophet (sazonalidade semanal + anual)', 'XGBoost com features de lag', 'Seasonal Naive (baseline)']
  CHURN_APPROACH               = classification
  MIN_IDENTIFIED_CUSTOMERS     = 1316
  CLUSTERING_VIABLE            = True




| Decisão | Valor | Justificativa |
|---|---|---|
| `CHURN_WINDOW_DAYS` || Janela de inatividade selecionada com base na análise de recência (Seção 7): janela mais curta que mantém a classe minoritária entre 20–50% para viabilizar classificação supervisionada. Se nenhuma janela atingir esse critério, adotar detecção de anomalia. |
| `FORECAST_PRODUCTS` || Produtos com ≥ 60 dias com pelo menos 1 venda (Seção 6.3). Produtos abaixo desse limiar não têm dados históricos suficientes para treinar um modelo individual com confiança estatística. |
| `FORECAST_MODEL_CANDIDATES` || Definido com base no comprimento e regularidade das séries (Seção 6): Prophet descartado para séries curtas (< 90 dias) ou com muitos zeros; XGBoost com features de lag (lag-1, lag-7, lag-14, rolling-7d) como candidato principal; Seasonal Naive como baseline. |
| `CHURN_APPROACH` || Se minority class ≥ 20%: classificação binária supervisionada (XGBoost/RF com scale_pos_weight ou SMOTE). Se minority class < 20%: detecção de anomalia (Isolation Forest / One-Class SVM). Decisão baseada na análise de balanceamento (Seção 7). |
| `MIN_IDENTIFIED_CUSTOMERS` || Total de clientes identificados (card ≠ anonymous) encontrados no dataset. Determina viabilidade do clustering RFM e do modelo de churn. |
| `CLUSTERING_VIABLE` || True se MIN_IDENTIFIED_CUSTOMERS ≥ 50. Abaixo desse limiar, o K-Means não produzirá clusters interpretáveis. Se False, a deve adotar segmentação por quintis como fallback e documentar a limitação. |